# Problem 3 - Data Parallelism in Pytorch

This notebook covers all four parts of Problem 3 using CIFAR-10 and a CIFAR-style ResNet-18 adapted from `kuangliu/pytorch-cifar`.

Implemented workflow:
- Part 1: single-GPU batch-size sweep with compute-only timing,
- Part 2: 1/2/4 GPU wall-clock timing and speedup with `torch.nn.DataParallel`,
- Part 3: compute-vs-communication breakdown,
- Part 4: all-reduce time and bandwidth utilization,
- local JSON/CSV result exports for every table plus a final aggregate JSON bundle.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

missing_packages = [
    package
    for package in ["torchvision", "pandas", "matplotlib"]
    if importlib.util.find_spec(package) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])


In [ ]:
import gc
import json
import math
import random
import time
from collections.abc import Iterable
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from IPython.display import display

SEED = 301
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    cudnn.benchmark = True

if not torch.cuda.is_available():
    raise RuntimeError(
        "Problem 3 requires a CUDA GPU. Run this notebook on Greene or another GPU node."
    )

PRIMARY_DEVICE = torch.device("cuda:0")
torch.cuda.set_device(PRIMARY_DEVICE)
VISIBLE_GPU_COUNT = torch.cuda.device_count()

RESULTS_DIR = Path("data/save/problem3_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_PATHS = {}

BANDWIDTH_GBPS = 200.0  # Replace with the correct Greene interconnect bandwidth before final submission.

print(f"Torch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Visible GPUs: {VISIBLE_GPU_COUNT}")
print(f"Using primary device: {torch.cuda.get_device_name(PRIMARY_DEVICE)}")
print(f"Results will be saved under: {RESULTS_DIR.resolve()}")


## CIFAR-10 Data Pipeline

The train/test loaders follow the homework prompt exactly: CIFAR-10, required transforms, `num_workers=2`, train batch size determined by the benchmark, and test batch size `100`.


In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2023, 0.1994, 0.2010)
DATA_ROOT = "./data/cifar10"
NUM_WORKERS = 2
TEST_BATCH_SIZE = 100

train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ]
)

test_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ]
)

train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=train_transform,
)
test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=test_transform,
)


def make_dataloaders(
    train_batch_size: int,
    test_batch_size: int = TEST_BATCH_SIZE,
    num_workers: int = NUM_WORKERS,
):
    common_loader_kwargs = {
        "num_workers": num_workers,
        "pin_memory": True,
        "persistent_workers": num_workers > 0,
    }
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=train_batch_size,
        shuffle=True,
        **common_loader_kwargs,
    )
    test_loader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=test_batch_size,
        shuffle=False,
        **common_loader_kwargs,
    )
    return train_loader, test_loader


print(f"Training examples: {len(train_dataset):,}")
print(f"Test examples: {len(test_dataset):,}")


## ResNet-18 Model

The model definition is adapted from the CIFAR-10 ResNet implementation used in `kuangliu/pytorch-cifar`, embedded directly in this notebook so the benchmark is self-contained.


In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_planes,
            planes,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(
            planes,
            planes,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes,
                    self.expansion * planes,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(self.expansion * planes),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block: type[BasicBlock], num_blocks: list[int], num_classes: int = 10):
        super().__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block: type[BasicBlock], planes: int, num_blocks: int, stride: int) -> nn.Sequential:
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for current_stride in strides:
            layers.append(block(self.in_planes, planes, current_stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = torch.flatten(out, 1)
        out = self.linear(out)
        return out


def build_resnet18(num_classes: int = 10) -> nn.Module:
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)


def model_gradient_bytes(model: nn.Module) -> int:
    return sum(parameter.numel() * parameter.element_size() for parameter in model.parameters())


model_preview = build_resnet18().to(PRIMARY_DEVICE)
parameter_count = sum(parameter.numel() for parameter in model_preview.parameters())
gradient_bytes = model_gradient_bytes(model_preview)
print(f"ResNet-18 parameter count: {parameter_count:,}")
print(f"Gradient payload per optimizer step: {gradient_bytes:,} bytes")
del model_preview
torch.cuda.empty_cache()


## Benchmark and Export Helpers

The helpers below power all four parts. Each benchmark run records both full wall-clock time and device-only time so later sections can reuse the same raw measurements.


In [ ]:
LEARNING_RATE = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
START_BATCH_SIZE = 32
BATCH_MULTIPLIER = 4
MAX_BATCH_SIZE = 65536
NUM_EPOCHS_PER_RUN = 2
GPU_SETUPS = [1, 2, 4]

criterion = nn.CrossEntropyLoss()


def synchronize(device: torch.device = PRIMARY_DEVICE) -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize(device)


def json_ready(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, torch.device):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    return value


def dataframe_records(df: Optional[pd.DataFrame]):
    if df is None or df.empty:
        return []
    safe_df = df.astype(object).where(pd.notna(df), None)
    return [json_ready(record) for record in safe_df.to_dict(orient="records")]


def save_records_json(name: str, records: list[dict]) -> Path:
    path = RESULTS_DIR / f"{name}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(json_ready(list(records)), f, indent=2)
    ARTIFACT_PATHS[name] = {"json": str(path)}
    print(f"Saved {path}")
    return path


def save_dataframe_outputs(name: str, df: pd.DataFrame):
    json_path = RESULTS_DIR / f"{name}.json"
    csv_path = RESULTS_DIR / f"{name}.csv"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(dataframe_records(df), f, indent=2)
    df.to_csv(csv_path, index=False)
    ARTIFACT_PATHS[name] = {"json": str(json_path), "csv": str(csv_path)}
    print(f"Saved {json_path}")
    print(f"Saved {csv_path}")
    return ARTIFACT_PATHS[name]


def save_problem3_bundle(bundle_name: str = "problem3_full_results") -> Path:
    def maybe_df(name: str) -> Optional[pd.DataFrame]:
        value = globals().get(name)
        return value if isinstance(value, pd.DataFrame) else None

    bundle = {
        "seed": SEED,
        "visible_gpu_count": VISIBLE_GPU_COUNT,
        "bandwidth_gbps": BANDWIDTH_GBPS,
        "artifact_paths": json_ready(ARTIFACT_PATHS),
        "part1_results": dataframe_records(maybe_df("part1_results_df")),
        "part2_results": dataframe_records(maybe_df("part2_results_df")),
        "part3_results": dataframe_records(maybe_df("part3_results_df")),
        "part4_results": dataframe_records(maybe_df("part4_results_df")),
    }
    path = RESULTS_DIR / f"{bundle_name}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(bundle, f, indent=2)
    ARTIFACT_PATHS[bundle_name] = {"json": str(path)}
    print(f"Saved {path}")
    return path


def requested_device_ids(gpu_count: int) -> list[int]:
    if gpu_count < 1:
        raise ValueError("gpu_count must be at least 1")
    if gpu_count > VISIBLE_GPU_COUNT:
        raise ValueError(f"Requested {gpu_count} GPUs but only {VISIBLE_GPU_COUNT} are visible.")
    return list(range(gpu_count))


def build_model_for_gpu_count(gpu_count: int) -> nn.Module:
    base_model = build_resnet18().to(PRIMARY_DEVICE)
    if gpu_count > 1:
        return nn.DataParallel(base_model, device_ids=requested_device_ids(gpu_count))
    return base_model


def make_batch_schedule(
    start: int = START_BATCH_SIZE,
    multiplier: int = BATCH_MULTIPLIER,
    max_batch_size: int = MAX_BATCH_SIZE,
) -> list[int]:
    batch_sizes = []
    current_batch_size = start
    while current_batch_size <= max_batch_size:
        batch_sizes.append(current_batch_size)
        current_batch_size *= multiplier
    return batch_sizes


def train_one_epoch_measurements(
    model: nn.Module,
    loader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> dict[str, float]:
    model.train()
    device_only_seconds = 0.0
    running_loss = 0.0
    steps = 0

    synchronize(device)
    wallclock_start = time.perf_counter()
    data_iterator = iter(loader)

    while True:
        try:
            inputs, targets = next(data_iterator)
        except StopIteration:
            break

        synchronize(device)
        step_start = time.perf_counter()

        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        synchronize(device)
        device_only_seconds += time.perf_counter() - step_start
        running_loss += float(loss.detach().item())
        steps += 1

    synchronize(device)
    wallclock_seconds = time.perf_counter() - wallclock_start
    return {
        "wallclock_seconds": wallclock_seconds,
        "device_only_seconds": device_only_seconds,
        "avg_loss": running_loss / max(steps, 1),
        "steps": steps,
    }


def empty_benchmark_record(
    batch_size_per_gpu: int,
    gpu_count: int,
    status: str,
    message: Optional[str] = None,
):
    return {
        "batch_size_per_gpu": batch_size_per_gpu,
        "gpu_count": gpu_count,
        "global_batch_size": batch_size_per_gpu * gpu_count,
        "status": status,
        "message": message,
        "epoch1_wallclock_seconds": None,
        "epoch2_wallclock_seconds": None,
        "epoch1_device_only_seconds": None,
        "epoch2_device_only_seconds": None,
        "epoch2_avg_loss": None,
        "steps": None,
    }


def run_two_epoch_benchmark(batch_size_per_gpu: int, gpu_count: int) -> dict[str, object]:
    if gpu_count > VISIBLE_GPU_COUNT:
        return empty_benchmark_record(
            batch_size_per_gpu,
            gpu_count,
            status="skipped_insufficient_gpus",
            message=f"Only {VISIBLE_GPU_COUNT} GPUs are visible.",
        )

    gc.collect()
    torch.cuda.empty_cache()

    train_loader = None
    test_loader = None
    model = None
    optimizer = None

    try:
        global_batch_size = batch_size_per_gpu * gpu_count
        train_loader, test_loader = make_dataloaders(train_batch_size=global_batch_size)
        model = build_model_for_gpu_count(gpu_count)
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=LEARNING_RATE,
            momentum=MOMENTUM,
            weight_decay=WEIGHT_DECAY,
        )

        history = []
        for epoch_index in range(NUM_EPOCHS_PER_RUN):
            stats = train_one_epoch_measurements(model, train_loader, optimizer, criterion, PRIMARY_DEVICE)
            history.append(stats)
            print(
                f"batch_size_per_gpu={batch_size_per_gpu} gpu_count={gpu_count} epoch={epoch_index + 1} "
                f"wallclock={stats['wallclock_seconds']:.2f}s device_only={stats['device_only_seconds']:.2f}s "
                f"loss={stats['avg_loss']:.4f}"
            )

        return {
            "batch_size_per_gpu": batch_size_per_gpu,
            "gpu_count": gpu_count,
            "global_batch_size": global_batch_size,
            "status": "ok",
            "message": None,
            "epoch1_wallclock_seconds": history[0]["wallclock_seconds"],
            "epoch2_wallclock_seconds": history[1]["wallclock_seconds"],
            "epoch1_device_only_seconds": history[0]["device_only_seconds"],
            "epoch2_device_only_seconds": history[1]["device_only_seconds"],
            "epoch2_avg_loss": history[1]["avg_loss"],
            "steps": history[1]["steps"],
        }
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return empty_benchmark_record(
            batch_size_per_gpu,
            gpu_count,
            status="oom",
            message="CUDA out of memory",
        )
    finally:
        del train_loader, test_loader, model, optimizer
        gc.collect()
        torch.cuda.empty_cache()


def run_single_gpu_sweep(batch_sizes: Iterable[int]) -> pd.DataFrame:
    records = []
    for batch_size_per_gpu in batch_sizes:
        print(f"\nRunning single-GPU benchmark for batch size {batch_size_per_gpu}...")
        record = run_two_epoch_benchmark(batch_size_per_gpu=batch_size_per_gpu, gpu_count=1)
        records.append(record)
        if record["status"] == "oom":
            print(f"Stopped after the first CUDA OOM at batch size {batch_size_per_gpu}.")
            break
    return pd.DataFrame(records)


## Part 1 - Single-GPU Baseline

The Part 1 sweep measures epoch-2 device-only time on exactly one GPU. This excludes DataLoader wait but includes CPU-to-GPU transfer, forward/backward, and the optimizer update.


In [ ]:
candidate_batch_sizes = make_batch_schedule()
part1_results_df = run_single_gpu_sweep(candidate_batch_sizes)
display(part1_results_df)
save_dataframe_outputs("part1_single_gpu_results", part1_results_df)

part1_successful_df = part1_results_df[part1_results_df["status"] == "ok"].copy()
if part1_successful_df.empty:
    raise RuntimeError("No batch size completed successfully on a single GPU.")

largest_feasible_batch_size = int(part1_successful_df["batch_size_per_gpu"].max())
fastest_row = part1_successful_df.sort_values("epoch2_device_only_seconds").iloc[0]

print(f"Largest feasible single-GPU batch size: {largest_feasible_batch_size}")
print(
    f"Fastest epoch-2 device-only time: {fastest_row['epoch2_device_only_seconds']:.2f} seconds "
    f"at batch size {int(fastest_row['batch_size_per_gpu'])}."
)


In [ ]:
plot_df = part1_successful_df.sort_values("batch_size_per_gpu")
ax = plot_df.plot(
    x="batch_size_per_gpu",
    y="epoch2_device_only_seconds",
    marker="o",
    figsize=(7, 4),
    legend=False,
)
ax.set_title("Problem 3 Part 1: Single-GPU Epoch-2 Device-Only Time")
ax.set_xlabel("Batch size per GPU")
ax.set_ylabel("Device-only training time (seconds)")
ax.set_xscale("log", base=4)
ax.grid(True, alpha=0.3)
plt.show()

part1_answer = "\n".join(
    [
        f"Largest feasible single-GPU batch size: {largest_feasible_batch_size}",
        (
            "Reported quantity: epoch-2 device-only time on one GPU, excluding DataLoader wait, "
            "but including CPU-to-GPU transfer, forward pass, backward pass, and optimizer update."
        ),
        (
            f"Fastest measured epoch-2 device-only time: {fastest_row['epoch2_device_only_seconds']:.2f} seconds "
            f"at batch size {int(fastest_row['batch_size_per_gpu'])}."
        ),
    ]
)
print(part1_answer)


## Part 2 - 1/2/4 GPU Wall-Clock Timing and Speedup

For each successful Part 1 batch size per GPU, this section benchmarks `1`, `2`, and `4` GPUs using the same per-GPU batch size. The `speedup` uses the full wall-clock 1-GPU time as the denominator, so the notebook is measuring weak scaling.


In [ ]:
if "part1_successful_df" not in globals() or part1_successful_df.empty:
    raise RuntimeError("Run Part 1 before running Part 2.")

part2_records = []
for batch_size_per_gpu in part1_successful_df["batch_size_per_gpu"].tolist():
    for gpu_count in GPU_SETUPS:
        print(
            f"\nRunning Part 2 benchmark for batch_size_per_gpu={batch_size_per_gpu}, gpu_count={gpu_count}..."
        )
        part2_records.append(
            run_two_epoch_benchmark(batch_size_per_gpu=batch_size_per_gpu, gpu_count=gpu_count)
        )

part2_results_df = pd.DataFrame(part2_records)
single_gpu_wallclock = (
    part2_results_df[
        (part2_results_df["gpu_count"] == 1) & (part2_results_df["status"] == "ok")
    ][["batch_size_per_gpu", "epoch2_wallclock_seconds"]]
    .rename(columns={"epoch2_wallclock_seconds": "single_gpu_wallclock_seconds"})
)
part2_results_df = part2_results_df.merge(single_gpu_wallclock, on="batch_size_per_gpu", how="left")
part2_results_df["speedup"] = part2_results_df.apply(
    lambda row: (
        row["single_gpu_wallclock_seconds"] / row["epoch2_wallclock_seconds"]
        if row["status"] == "ok"
        and pd.notna(row["single_gpu_wallclock_seconds"])
        and pd.notna(row["epoch2_wallclock_seconds"])
        and row["epoch2_wallclock_seconds"] > 0
        else None
    ),
    axis=1,
)

save_dataframe_outputs("part2_speedup_table", part2_results_df)
display(part2_results_df)

part2_answer = "\n".join(
    [
        "Table 1 is the full wall-clock benchmark, including DataLoader wait, CPU-to-GPU transfer, computation, and communication.",
        "Because the batch size per GPU stays fixed while the total batch size grows with GPU count, this notebook is measuring weak scaling.",
        "Under strong scaling, the total batch size would stay fixed and the reported speedup would typically look better than the weak-scaling result here.",
    ]
)
print(part2_answer)


## Part 3 - Compute vs Communication Time

This section follows the homework hint and uses the Part 1 single-GPU device-only timing to estimate compute time. For each successful `2`-GPU or `4`-GPU run:

- `compute_time_seconds = part1_single_gpu_device_only_seconds / gpu_count`
- `communication_time_seconds = multi_gpu_device_only_seconds - compute_time_seconds`

Small negative communication values are clipped to `0` and reported as measurement noise.


In [ ]:
if "part2_results_df" not in globals():
    raise RuntimeError("Run Part 2 before running Part 3.")

part1_compute_reference = part1_successful_df[
    ["batch_size_per_gpu", "epoch2_device_only_seconds"]
].rename(columns={"epoch2_device_only_seconds": "single_gpu_device_only_seconds"})

part3_results_df = part2_results_df[part2_results_df["gpu_count"].isin([2, 4])].copy()
part3_results_df = part3_results_df.merge(part1_compute_reference, on="batch_size_per_gpu", how="left")
part3_results_df["compute_time_seconds"] = part3_results_df.apply(
    lambda row: (
        row["single_gpu_device_only_seconds"] / row["gpu_count"]
        if row["status"] == "ok" and pd.notna(row["single_gpu_device_only_seconds"])
        else None
    ),
    axis=1,
)
part3_results_df["raw_communication_seconds"] = part3_results_df.apply(
    lambda row: (
        row["epoch2_device_only_seconds"] - row["compute_time_seconds"]
        if row["status"] == "ok"
        and pd.notna(row["epoch2_device_only_seconds"])
        and pd.notna(row["compute_time_seconds"])
        else None
    ),
    axis=1,
)
part3_results_df["measurement_noise_seconds"] = part3_results_df["raw_communication_seconds"].apply(
    lambda value: (
        abs(value)
        if value is not None and pd.notna(value) and value < 0
        else 0.0
        if value is not None and pd.notna(value)
        else None
    )
)
part3_results_df["communication_time_seconds"] = part3_results_df["raw_communication_seconds"].apply(
    lambda value: (
        max(value, 0.0)
        if value is not None and pd.notna(value)
        else None
    )
)

save_dataframe_outputs("part3_compute_comm_table", part3_results_df)
display(part3_results_df)


## Part 4 - All-reduce Time and Bandwidth Utilization

Assume PyTorch `DataParallel` uses ring all-reduce, with:

- `allreduce_bytes_per_iteration = 2 * (N - 1) / N * gradient_bytes`
- `ideal_allreduce_time_per_iteration = allreduce_bytes_per_iteration / peak_bandwidth_bytes_per_second`
- `bandwidth_utilization = epoch_allreduce_bytes / (measured_communication_time_epoch * peak_bandwidth_bytes_per_second)`

Set `BANDWIDTH_GBPS` near the top of the notebook to the correct Greene value before using the final Table 3 numbers.


In [ ]:
if "part3_results_df" not in globals():
    raise RuntimeError("Run Part 3 before running Part 4.")

gradient_bytes = model_gradient_bytes(build_resnet18())
peak_bandwidth_bytes_per_second = BANDWIDTH_GBPS * 1_000_000_000

part4_results_df = part3_results_df.copy()
part4_results_df["gradient_bytes"] = gradient_bytes
part4_results_df["allreduce_bytes_per_iteration"] = part4_results_df["gpu_count"].apply(
    lambda gpu_count: (
        2 * (gpu_count - 1) / gpu_count * gradient_bytes
        if pd.notna(gpu_count) and gpu_count > 1
        else None
    )
)
part4_results_df["allreduce_bytes_per_epoch"] = part4_results_df.apply(
    lambda row: (
        row["allreduce_bytes_per_iteration"] * row["steps"]
        if row["status"] == "ok"
        and pd.notna(row["allreduce_bytes_per_iteration"])
        and pd.notna(row["steps"])
        else None
    ),
    axis=1,
)
part4_results_df["ideal_allreduce_time_epoch_seconds"] = part4_results_df[
    "allreduce_bytes_per_epoch"
].apply(
    lambda value: (
        value / peak_bandwidth_bytes_per_second
        if value is not None and pd.notna(value)
        else None
    )
)
part4_results_df["bandwidth_utilization"] = part4_results_df.apply(
    lambda row: (
        row["allreduce_bytes_per_epoch"]
        / (row["communication_time_seconds"] * peak_bandwidth_bytes_per_second)
        if row["status"] == "ok"
        and pd.notna(row["allreduce_bytes_per_epoch"])
        and pd.notna(row["communication_time_seconds"])
        and row["communication_time_seconds"] > 0
        else None
    ),
    axis=1,
)

save_dataframe_outputs("part4_bandwidth_table", part4_results_df)
display(part4_results_df)

bundle_path = save_problem3_bundle()
print(f"Saved aggregate bundle to: {bundle_path}")


## Submission Notes

- The notebook writes JSON/CSV outputs to `data/save/problem3_results/` after each part.
- `part1_single_gpu_results.json` and `.csv` store the Part 1 sweep.
- `part2_speedup_table.json` and `.csv` store Table 1.
- `part3_compute_comm_table.json` and `.csv` store Table 2.
- `part4_bandwidth_table.json` and `.csv` store Table 3.
- `problem3_full_results.json` stores the full benchmark bundle in one file.
- If the current allocation has fewer than 4 visible GPUs, the notebook records skipped rows instead of hard-failing.
